# Chat SDG Pipeline

Generates synthetic training data for telecom intent classification in Thai
using the sdg_hub framework. Reads existing seed data from JSON files and
generates the defined number of new examples per intent (default=20).

In [1]:
%pip install litellm==1.82.6 sdg_hub==0.9.1

Note: you may need to restart the kernel to use updated packages.


## Imports

Standard library imports used throughout the script. `argparse` is for CLI argument parsing, `json` for reading seed files and writing output, `time` for elapsed time tracking and `pathlib.Path` for all file path handling.

In [2]:
import argparse
import json
import os
import sys
import time
from pathlib import Path

## Intent Descriptions

A dictionary mapping each intent key to a plain English description of what the customer is trying to do. These descriptions are injected into the LLM prompt so the model understands the target intent when generating synthetic examples. The descriptions are loaded from `intent_descriptions.json` so that `run.py` and this notebook always stay in sync.

In [3]:
_INTENT_DESC_PATH = Path("intent_descriptions.json")
with open(_INTENT_DESC_PATH, encoding="utf-8") as _f:
    INTENT_DESCRIPTIONS: dict[str, str] = json.load(_f)

## load_seed_data

Loads `training_dataset.json` and groups the examples by `intent`. Returns a dict mapping each intent name to its list of seed examples.

In [4]:
def load_seed_data(dataset_path: Path) -> dict[str, list[dict]]:
    """Load training_dataset.json and group examples by intent."""
    with open(dataset_path, encoding="utf-8") as fh:
        data = json.load(fh)
    seeds: dict[str, list[dict]] = {}
    for ex in data:
        intent = ex.get("intent", "")
        if intent:
            seeds.setdefault(intent, []).append(ex)
    return seeds

## select_seed_examples

Picks a diverse subset of up to `max_seeds` (default 5) seed examples. It groups examples by the length of their `session_history` then takes one from each length bucket in round robin order. After selection it truncates any history longer than 5 messages to the last 5 (sliding window). This gives the LLM a mix of short and long conversation contexts as few shot examples.

In [5]:
def select_seed_examples(examples: list[dict], max_seeds: int = 5) -> list[dict]:
    """Pick a diverse subset of seed examples (varied history lengths)."""
    by_len = {}
    for ex in examples:
        h_len = len(ex.get("session_history", []))
        by_len.setdefault(h_len, []).append(ex)

    selected = []
    # Take one from each history-length bucket, round-robin
    buckets = sorted(by_len.keys())
    idx = 0
    while len(selected) < max_seeds and idx < len(examples):
        for b in buckets:
            if by_len[b] and len(selected) < max_seeds:
                selected.append(by_len[b].pop(0))
        idx += 1

    # Truncate seed histories to max 5 messages (sliding window)
    for ex in selected:
        if len(ex.get("session_history", [])) > 5:
            ex["session_history"] = ex["session_history"][-5:]

    return selected

## build_prompt

Constructs the chat messages sent to the LLM. It loads the YAML template from `prompts/generate_telco_chat.yaml` (which defines system and user messages with placeholder variables) and does simple string replacement for `{{ intent }}`, `{{ intent_description }}`, `{{ seed_examples }}`, `{{ num_examples }}` and `{{ num_seed }}`. The result is a list of `{role, content}` dicts ready for a chat completion call.

Note: `yaml` is imported inside the function rather than at the top of the script.

In [6]:
_NOTEBOOK_DIR = Path(".").resolve()

def build_prompt(intent: str, seed_examples: list[dict], num_examples: int = 20) -> list[dict]:
    """Build chat messages for the LLM call."""
    prompt_path = _NOTEBOOK_DIR / "prompts" / "generate_telco_chat.yaml"

    # Read the YAML template manually (avoid heavy deps)
    import yaml
    with open(prompt_path) as f:
        template_messages = yaml.safe_load(f)

    # Simple Jinja2-style substitution
    seed_json = json.dumps(seed_examples, ensure_ascii=False, indent=2)
    description = INTENT_DESCRIPTIONS.get(intent, f"Intent: {intent}")

    messages = []
    for msg in template_messages:
        content = msg["content"]
        content = content.replace("{{ intent }}", intent)
        content = content.replace("{{ intent_description }}", description)
        content = content.replace("{{ seed_examples }}", seed_json)
        content = content.replace("{{ num_examples }}", str(num_examples))
        content = content.replace("{{ num_seed }}", str(len(seed_examples)))
        messages.append({"role": msg["role"], "content": content})

    return messages

## parse_json_output

Extracts a JSON array from raw LLM text output. First strips markdown code fences if present. Then finds the first `[` and last `]` and tries to parse that substring as JSON. If the result is a list it is returned directly. On any failure it prints a warning and returns an empty list (the comment mentions line by line recovery but the current implementation does not attempt it).

In [7]:
def parse_json_output(text: str) -> list[dict]:
    """Extract a JSON array from the LLM output text."""
    # Try direct parse first
    text = text.strip()
    if text.startswith("```"):
        # Remove markdown code fences
        lines = text.split("\n")
        lines = [l for l in lines if not l.strip().startswith("```")]
        text = "\n".join(lines).strip()

    # Find the JSON array
    start = text.find("[")
    end = text.rfind("]")
    if start != -1 and end != -1:
        json_str = text[start : end + 1]
        try:
            data = json.loads(json_str)
            if isinstance(data, list):
                return data
        except json.JSONDecodeError:
            pass

    # Fallback: try parsing line by line for individual JSON objects
    print("  WARNING: Could not parse full JSON array, attempting recovery...")
    return []

## validate_examples

Post processing step applied to every generated batch regardless of mode. For each example it:

- Drops anything that is not a dict or is missing `user_message`
- Forces `intent` to the correct value (overwriting whatever the model produced)
- Ensures `session_history` is a list of `{role, content}` dicts where role is either `user` or `assistant`
- Trims history to the last 5 messages (same sliding window constraint the prompt asks the model to follow)
- Builds a clean output dict with only the three expected keys: `user_message`, `intent`, `session_history`

In [8]:
def validate_examples(examples: list[dict], intent: str) -> list[dict]:
    """Validate and fix generated examples."""
    valid = []
    for ex in examples:
        if not isinstance(ex, dict):
            continue
        # Ensure required fields
        if "user_message" not in ex:
            continue
        ex["intent"] = intent  # Force correct intent
        if "session_history" not in ex:
            ex["session_history"] = []
        # Validate session_history structure
        if not isinstance(ex["session_history"], list):
            ex["session_history"] = []
        cleaned_history = []
        for msg in ex["session_history"]:
            if isinstance(msg, dict) and "role" in msg and "content" in msg:
                if msg["role"] in ("user", "assistant"):
                    cleaned_history.append({
                        "role": msg["role"],
                        "content": str(msg["content"]),
                    })
        # Enforce sliding window: keep only the last 5 messages
        if len(cleaned_history) > 5:
            cleaned_history = cleaned_history[-5:]
        ex["session_history"] = cleaned_history
        valid.append({
            "user_message": str(ex["user_message"]),
            "intent": intent,
            "session_history": ex["session_history"],
        })
    return valid

## generate_with_litellm (Standalone Backend)

The standalone generation path. Calls `build_prompt` to get chat messages, then sends them to `litellm.completion` with `max_tokens=16384`, `temperature=0.8` and a 30 minute timeout (for slow local models). Optionally passes `api_base` and `api_key` for self hosted or keyed endpoints. The raw text response is run through `parse_json_output` and returned.

Note: `litellm` is imported inside the function so the module is only needed when this path is actually used.

In [9]:
def generate_with_litellm(
    intent: str,
    seed_examples: list[dict],
    num_examples: int,
    model: str,
    api_base: str | None = None,
    api_key: str | None = None,
) -> list[dict]:
    """Generate examples using litellm directly."""
    import litellm

    messages = build_prompt(intent, seed_examples, num_examples)

    kwargs = {
        "model": model,
        "messages": messages,
        "max_tokens": 8192,
        "temperature": 0.8,
        "timeout": 3600,
    }
    if api_base:
        kwargs["api_base"] = api_base
    if api_key:
        kwargs["api_key"] = api_key

    response = litellm.completion(**kwargs)
    text = response.choices[0].message.content
    return parse_json_output(text)

## generate_with_flow (sdg_hub Backend)

The alternative generation path using the `sdg_hub` library. Loads `flow.yaml` which defines a three block pipeline: `PromptBuilderBlock` (renders the same YAML prompt template), `LLMChatBlock` (calls the model with `async_mode: true`) and `LLMResponseExtractorBlock` (pulls text content from the response). All intents are passed as rows in a single pandas DataFrame and `flow.generate` runs them with up to 5 concurrent requests.

The function returns `dict[str, str]` mapping intent name to raw text output. Parsing and validation happen in the caller.

In [10]:
def generate_with_flow(
    intents_data: list[dict],
    model: str,
    api_base: str | None = None,
    api_key: str | None = None,
) -> dict[str, str]:
    """Generate using sdg_hub Flow (all intents in one batch)."""
    import pandas as pd
    from sdg_hub import Flow

    flow_path = _NOTEBOOK_DIR / "flow.yaml"
    flow = Flow.from_yaml(str(flow_path))

    model_config = {"model": model}
    if api_base:
        model_config["api_base"] = api_base
    if api_key:
        model_config["api_key"] = api_key
    flow.set_model_config(**model_config)

    df = pd.DataFrame(intents_data)
    result = flow.generate(df, max_concurrency=5)

    outputs = {}
    for _, row in result.iterrows():
        intent = row["intent"]
        content = row.get("extract_output_content", "")
        outputs[intent] = content
    return outputs

## run_standalone

Extracted from the standalone branch of `main()`. Processes intents one at a time in a sequential loop. For each intent it selects seeds, calls `generate_with_litellm`, validates the output, warns if fewer than requested examples passed validation and writes each intent to its own JSON file. Exceptions are caught per intent so a failure on one intent does not stop the rest.

In [11]:
def run_standalone(args, seeds, intents_to_generate, output_dir):
    all_results = {}
    for i, intent in enumerate(intents_to_generate, 1):
        print(f"[{i}/{len(intents_to_generate)}] Generating: {intent}")
        seed_subset = select_seed_examples(seeds[intent], args.max_seeds)
        print(f"  Using {len(seed_subset)} seed examples")

        start = time.time()
        try:
            generated = generate_with_litellm(
                intent=intent,
                seed_examples=seed_subset,
                num_examples=args.num_examples,
                model=args.model,
                api_base=args.api_base,
                api_key=args.api_key,
            )
            validated = validate_examples(generated, intent)
            elapsed = time.time() - start
            print(f"  Generated {len(validated)} valid examples ({elapsed:.1f}s)")

            if len(validated) < args.num_examples:
                print(f"  WARNING: Only {len(validated)}/{args.num_examples} valid examples")

            all_results[intent] = validated

            # Save individual intent file
            out_file = output_dir / f"{intent}.json"
            with open(out_file, "w", encoding="utf-8") as f:
                json.dump(validated, f, ensure_ascii=False, indent=2)
            print(f"  Saved to {out_file}")

        except Exception as e:
            print(f"  ERROR: {e}")
            all_results[intent] = []

## run_flow

Extracted from the flow branch of `main()`. Starts by checking if `sdg_hub` and `pandas` can be imported, exiting with an install hint if not. Then builds a list of row dicts (one per intent) with `intent`, `intent_description`, `seed_examples` as a JSON string, `num_examples` and `num_seed` as strings. Passes them all to `generate_with_flow` which runs them concurrently. Each returned raw text is then parsed and validated the same way as standalone. Any exception from the flow run exits the script entirely (unlike standalone which continues past per intent errors).

In [12]:
def run_flow(args, seeds, intents_to_generate, output_dir):
    try:
        import pandas as pd  # noqa: F401
        from sdg_hub import Flow  # noqa: F401
    except ImportError:
        print("ERROR: sdg_hub not installed. Run: pip install sdg-hub")
        print("       Or use --mode standalone for litellm-only mode.")
        sys.exit(1)

    # Prepare input DataFrame
    intents_data = []
    for intent in intents_to_generate:
        seed_subset = select_seed_examples(seeds[intent], args.max_seeds)
        intents_data.append({
            "intent": intent,
            "intent_description": INTENT_DESCRIPTIONS.get(intent, intent),
            "seed_examples": json.dumps(seed_subset, ensure_ascii=False, indent=2),
            "num_examples": str(args.num_examples),
            "num_seed": str(len(seed_subset)),
        })

    print("Running sdg_hub flow...")
    start = time.time()
    try:
        outputs = generate_with_flow(
            intents_data=intents_data,
            model=args.model,
            api_base=args.api_base,
            api_key=args.api_key,
        )
        elapsed = time.time() - start
        print(f"Flow completed in {elapsed:.1f}s")

        for intent, raw_text in outputs.items():
            generated = parse_json_output(raw_text)
            validated = validate_examples(generated, intent)
            print(f"  {intent}: {len(validated)} valid examples")

            out_file = output_dir / f"{intent}.json"
            with open(out_file, "w", encoding="utf-8") as f:
                json.dump(validated, f, ensure_ascii=False, indent=2)

    except Exception as e:
        print(f"ERROR: Flow execution failed: {e}")
        sys.exit(1)

## print_summary

Extracted from the tail end of `main()`. Runs after either mode completes. Globs all `*.json` files in `output_dir` (not just the intents generated in this run), loads each one, counts examples and prints per file and total counts.

In [13]:
def print_summary(output_dir):
    print("\n" + "=" * 60)
    print("GENERATION COMPLETE")
    print("=" * 60)
    total = 0
    for f in sorted(output_dir.glob("*.json")):
        with open(f) as fh:
            data = json.load(fh)
            total += len(data)
            print(f"  {f.name}: {len(data)} examples")
    print(f"\n  Total: {total} examples in {output_dir}")

## Pipeline Configuration

Edit the values below before running the pipeline. These replace the CLI arguments used in `run.py`.

In [14]:
MODE = "flow"                          # "standalone" or "flow"
MODEL = "gemini/gemma-4-31b-it"            # litellm format: provider/model
API_BASE = None                              # not needed for gemini
API_KEY = os.environ.get("GEMINI_API_KEY", "")  # set your key here or via GEMINI_API_KEY env var
OUTPUT = "None"                                # output directory (default: ../generated)
NUM_EXAMPLES = 5                             # number of examples to generate per intent
MAX_SEEDS = 5                                # max seed examples to include in prompt
INTENTS = None                               # list of intents to generate, e.g. ["MOBILE_BILLING_CHECK_DUE_DATE"], or None for all

os.environ["GEMINI_API_KEY"] = API_KEY

## main

Orchestrates the pipeline. Seed data is loaded from `samples/training_dataset.json` and generated output goes to `generated/`. After loading seeds it filters to only requested intents (or all if none specified) and exits if none remain. Delegates to `run_standalone`, `run_flow` and `print_summary` defined in the cells above.

In [15]:
def main():
    from types import SimpleNamespace
    args = SimpleNamespace(
        mode=MODE,
        model=MODEL,
        api_base=API_BASE,
        api_key=API_KEY,
        output=OUTPUT,
        num_examples=NUM_EXAMPLES,
        max_seeds=MAX_SEEDS,
        intents=INTENTS,
    )

    # Paths
    repo_root = _NOTEBOOK_DIR.parent
    dataset_path = repo_root / "samples" / "training_dataset.json"
    output_dir = Path(args.output) if args.output else repo_root / "generated"
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load seed data
    print("Loading seed data...")
    seeds = load_seed_data(dataset_path)
    print(f"  Found {len(seeds)} intents: {', '.join(seeds.keys())}")

    # Filter intents if specified
    intents_to_generate = args.intents if args.intents else list(seeds.keys())
    intents_to_generate = [i for i in intents_to_generate if i in seeds]

    if not intents_to_generate:
        print("ERROR: No valid intents found to generate.")
        sys.exit(1)

    print(f"\nGenerating {args.num_examples} examples each for {len(intents_to_generate)} intents")
    print(f"Model: {args.model}")
    print(f"Output: {output_dir}\n")

    if args.mode == "standalone":
        run_standalone(args, seeds, intents_to_generate, output_dir)
    elif args.mode == "flow":
        run_flow(args, seeds, intents_to_generate, output_dir)

    print_summary(output_dir)

## Entrypoint

Standard Python entrypoint guard. When run as a script (`python run.py ...`) it calls `main()`. In this notebook form the cells above define all the functions and constants but do not execute the pipeline unless you call `main()` yourself.

In [16]:
%pip install nest_asyncio 

import nest_asyncio; 
nest_asyncio.apply()

Note: you may need to restart the kernel to use updated packages.


In [17]:
main()

Loading seed data...
  Found 16 intents: MOBILE_SIM_ACTIVATION_POSTPAID, MOBILE_SIM_ACTIVATION_PREPAID, MOBILE_SIM_REPLACEMENT_POSPAID, MOBILE_SIM_REPLACEMENT_PREPAID, MOBILE_ROAMING_CHECK_DATA_CURRENT, MOBILE_ROAMING_CHECK_PAYMENT_POSTPAID, MOBILE_USAGE_CHECK_DATA_CURRENT, MOBILE_USAGE_COMPARE_DATA_PLAN, MOBILE_BILLING_CHECK_DUE_DATE, MOBILE_BILLING_PAYMENT_STATUS_POSTPAID, MOBILE_BILLING_PAYMENT_STATUS_PREPAID, MOBILE_NETWORK_CHECK_NO_SIGNAL, MOBILE_PACKAGE_CHECK_DATA_CURRENT, MOBILE_PACKAGE_CHECK_DATA_HISTORY, MOBILE_VAS_SUBSCRIBE_DATA_HISTORY, MOBILE_VAS_SUBSCRIBE_DATA_SPECIFIC_MONTH

Generating 5 examples each for 16 intents
Model: gemini/gemma-4-31b-it
Output: None



/Users/bbalasub/Desktop/ais-ai-th/sdg/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running sdg_hub flow...


[21:30:57] INFO     Loading flow from: /Users/bbalasub/Desktop/ais-ai-th/sdg/pipeline/flow.yaml ]8;id=15858707;file:///Users/bbalasub/Desktop/ais-ai-th/sdg/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/serialization.py\serialization.py]8;;\:]8;id=15858708;file:///Users/bbalasub/Desktop/ais-ai-th/sdg/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/serialization.py#61\61]8;;\

[21:30:57] INFO     Auto-detected 1 LLM blocks for configuration: ['generate_examples']         ]8;id=15858715;file:///Users/bbalasub/Desktop/ais-ai-th/sdg/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=15858716;file:///Users/bbalasub/Desktop/ais-ai-th/sdg/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py#242\242]8;;\

           INFO     Successfully configured 1 LLM blocks with: model: 'gemini/gemma-4-31b-it',  ]8;id=15858722;file:///Users/bbalasub/Desktop/ais-ai-th/sdg/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=15858723;file:///Users/bbalasub/Desktop/ais-ai-th/sdg/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py#300\300]8;;\
                    api_key: (redacted)                                                                            

... (17 outputs left out)

           INFO     Block 'extract_output' completed successfully: 16 samples, 8 columns           ]8;id=15858797;file:///Users/bbalasub/Desktop/ais-ai-th/sdg/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=15858798;file:///Users/bbalasub/Desktop/ais-ai-th/sdg/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py#270\270]8;;\

╭────────────────────────────── Telco Chat Training Data Generation Flow - Complete ──────────────────────────────╮
│                                        Flow Execution Summary                                                   │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓           │
│ ┃ Block Name           ┃ Type            ┃   Duration ┃     Rows     ┃     Columns     ┃   Status   ┃           │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩           │
│ │ build_generation_pr… │ PromptBuilderB… │      0.01s │   16 → 16    │       +1        │     ✓      │           │
│ │ generate_examples    │ LLMChatBlock    │    260.08s │   16 → 16    │       +1        │     ✓      │           │
│ │ extract_output       │ LLMResponseExt… │      0.01s │   16 → 16    │       +1        │     ✓      │           │
│ ├──────────────────────┼─────────────────┼────────────┼──────────────┼─────────────────┼────────────┤           │
│ │ TOTAL                │ 3 blocks        │    260.10s │   16 final   │     8 final     │    3/3     │           │
│ └──────────────────────┴─────────────────┴────────────┴──────────────┴─────────────────┴────────────┘           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Flow 'Telco Chat Training Data Generation Flow' completed successfully: 16     ]8;id=15858804;file:///Users/bbalasub/Desktop/ais-ai-th/sdg/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=15858805;file:///Users/bbalasub/Desktop/ais-ai-th/sdg/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py#671\671]8;;\
                    final samples, 8 final columns                                                                 

Flow completed in 260.1s
  MOBILE_SIM_ACTIVATION_POSTPAID: 5 valid examples
  MOBILE_SIM_ACTIVATION_PREPAID: 5 valid examples
  MOBILE_SIM_REPLACEMENT_POSPAID: 5 valid examples
  MOBILE_SIM_REPLACEMENT_PREPAID: 5 valid examples
  MOBILE_ROAMING_CHECK_DATA_CURRENT: 5 valid examples
  MOBILE_ROAMING_CHECK_PAYMENT_POSTPAID: 5 valid examples
  MOBILE_USAGE_CHECK_DATA_CURRENT: 5 valid examples
  MOBILE_USAGE_COMPARE_DATA_PLAN: 5 valid examples
  MOBILE_BILLING_CHECK_DUE_DATE: 5 valid examples
  MOBILE_BILLING_PAYMENT_STATUS_POSTPAID: 5 valid examples
  MOBILE_BILLING_PAYMENT_STATUS_PREPAID: 5 valid examples
  MOBILE_NETWORK_CHECK_NO_SIGNAL: 5 valid examples
  MOBILE_PACKAGE_CHECK_DATA_CURRENT: 5 valid examples
  MOBILE_PACKAGE_CHECK_DATA_HISTORY: 5 valid examples
  MOBILE_VAS_SUBSCRIBE_DATA_HISTORY: 5 valid examples
  MOBILE_VAS_SUBSCRIBE_DATA_SPECIFIC_MONTH: 5 valid examples

GENERATION COMPLETE
  MOBILE_BILLING_CHECK_DUE_DATE.json: 5 examples
  MOBILE_BILLING_PAYMENT_STATUS_POSTPAID.json: